# S2.2 — Spark Execution Model

## The 4-Level Hierarchy


| Level | Triggered by | Count |
|-------|-------------|-------|
| Application | SparkSession start | 1 per notebook |
| Job | Each ACTION (count, show, write) | 1 per action |
| Stage | Separated by SHUFFLE | 2+ per job |
| Task | One per partition | 8 tasks = 8 partitions |

## Transformations vs Actions

| Type | Examples | Executes? |
|------|---------|-----------|
| Transformation | filter, groupBy, select, range | ❌ No — lazy |
| Action | count, show, collect, write | ✅ Yes — triggers everything |

## Rule
**As long as result is a DataFrame = Transformation = Lazy = No execution**  
**When result leaves Spark (number/rows/file) = Action = Execute NOW**

## Why "Lazy"?
Spark waits to see the FULL plan before executing.  
Then Catalyst Optimizer reorganises steps for efficiency.  
Lazy = smart, not slow. Full coverage in S2.4 + S2.9.

## From our S2.1 Query Profile
- Job triggered by: df.count() ← ACTION
- Stage 1: Range → Aggregate (before shuffle)
- Stage 2: Shuffle → Aggregate (after shuffle = new stage)
- Tasks: 8 per stage (one per partition)